稳定版一维 Softmax

In [1]:
import numpy as np
import math

def normlize(Z):
    res = np.zeros_like(Z, dtype=float)
    res = Z - max(Z)
    return res

def onedim_softmax(Z):
    Z = normlize(Z)
    res = np.zeros((len(Z),), dtype=float)
    sum1 = sum(np.exp(Z))
    for i in range(len(Z)):
        res[i] = np.exp(Z[i])/sum1
    return res

def logsumeps(Z):
    max = np.max(Z)
    Z = normlize(Z)
    ln = np.log(sum(np.exp(Z)))
    res = max + ln
    return res

z1 = np.array([2., 1., 0.])
z2 = np.array([1000., 999., 998.])
z3 = np.array([0., 0., 0.])
print("z1:", onedim_softmax(z1))
print("z2:", onedim_softmax(z2))
print("z3:", onedim_softmax(z3))

z1: [0.66524096 0.24472847 0.09003057]
z2: [0.66524096 0.24472847 0.09003057]
z3: [0.33333333 0.33333333 0.33333333]


自动验收

In [2]:
def residual(Z):
    residual1 = np.abs(np.sum(onedim_softmax(Z)) - 1)
    return residual1

print("残差：")
print("z1:",residual(z1),"PASS" if residual(z1) < 1e-12 else "FAIL")
print("z2:",residual(z2),"PASS" if residual(z2) < 1e-12 else "FAIL")
print("z3:",residual(z3),"PASS" if residual(z3) < 1e-12 else "FAIL")

print("z1 与 z2 概率的最大绝对差：")
print(np.max(np.abs(onedim_softmax(z1) - onedim_softmax(z2))), "PASS" if np.max(np.abs(onedim_softmax(z1) - onedim_softmax(z2))) < 1e-12 else "FAIL")

print("每组概率是否全部有限：")
print("z1:","PASS" if np.all(np.isfinite(onedim_softmax(z1))) else "FAIL")
print("z2:","PASS" if np.all(np.isfinite(onedim_softmax(z2))) else "FAIL")
print("z3:","PASS" if np.all(np.isfinite(onedim_softmax(z3))) else "FAIL")

print("每组概率是否全部非负：")
print("z1:", "PASS" if np.all(onedim_softmax(z1) >= 0) else "FAIL")
print("z2:", "PASS" if np.all(onedim_softmax(z2) >= 0) else "FAIL")
print("z3:", "PASS" if np.all(onedim_softmax(z3) >= 0) else "FAIL")


残差：
z1: 1.1102230246251565e-16 PASS
z2: 1.1102230246251565e-16 PASS
z3: 0.0 PASS
z1 与 z2 概率的最大绝对差：
0.0 PASS
每组概率是否全部有限：
z1: PASS
z2: PASS
z3: PASS
每组概率是否全部非负：
z1: PASS
z2: PASS
z3: PASS


从 logits 直接计算稳定交叉熵

In [3]:
import numpy as np

def normlize(Z):
    res = np.zeros_like(Z, dtype=float)
    res = Z - np.max(Z)
    return res

def onedim_softmax(Z):
    Z = normlize(Z)
    res = np.zeros((len(Z),), dtype=float)
    sum1 = sum(np.exp(Z))
    for i in range(len(Z)):
        res[i] = np.exp(Z[i])/sum1
    return res

def logsumeps(Z):
    max = np.max(Z)
    Z = normlize(Z)
    ln = np.log(sum(np.exp(Z)))
    res = max + ln
    return res

def LSE(Z,target):
    lse = logsumeps(Z)
    Z_true = Z[target]
    L = lse - Z_true
    return L

z1 = np.array([2., 1., 0.]); target1 = 0
z2 = np.array([2., 1., 0.]); target2 = 2
z3 = np.array([1000., 999., 998.]); target3 = 0
z4 = np.array([1000., 0., -1000.]); target4 = 2

print("z1, t0")
print("理论值：", 0.4076059644443804)
print("实际值：", LSE(z1, target1))
print("普通softmax：", -np.log(onedim_softmax(z1)[target1]), " ", "PASS" if np.abs(LSE(z1, target1) - (-np.log(onedim_softmax(z1)[target1]))) <= 1e-12 else "FAIL")
print("绝对残差：", np.abs(0.4076059644443804 - LSE(z1, target1)), " ", "PASS" if np.abs(0.4076059644443804 - LSE(z1, target1)) <= 1e-12 else "FAIL")

print("z1, t2")
print("理论值：", 2.407606)
print("实际值：", LSE(z2, target2))
print("绝对残差：", np.abs(2.407606 - LSE(z2, target2)), " ", "PASS" if np.abs(2.407606 - LSE(z2, target2)) <= 1e-6 else "FAIL")

print("z2, t0")
print("理论值：", 0.4076059644443804)
print("实际值：", LSE(z3, target3))
print("绝对残差：", np.abs(0.4076059644443804 - LSE(z3, target3)), " ", "PASS" if np.abs(0.4076059644443804 - LSE(z3, target3)) <= 1e-12 else "FAIL")

print("z4, t2")
print("理论值：", 2000)
print("实际值：", LSE(z4, target4))
print("绝对残差：", np.abs(2000 - LSE(z4, target4)), " ", "PASS" if np.abs(2000 - LSE(z4, target4)) <= 1e-12 else "FAIL")
print("是否都有限：", "PASS" if np.isfinite(LSE(z4, target4)) else "FAIL")



z1, t0
理论值： 0.4076059644443804
实际值： 0.4076059644443806
普通softmax： 0.40760596444438046   PASS
绝对残差： 1.6653345369377348e-16   PASS
z1, t2
理论值： 2.407606
实际值： 2.4076059644443806
绝对残差： 3.555561933765716e-08   PASS
z2, t0
理论值： 0.4076059644443804
实际值： 0.40760596444442854
绝对残差： 4.8128168117500536e-14   PASS
z4, t2
理论值： 2000
实际值： 2000.0
绝对残差： 0.0   PASS
是否都有限： PASS


扩展到 batch

In [4]:
import numpy as np

def batch_normlize(Z):
    Z_norm = np.zeros_like(Z, dtype=float)
    max = np.zeros(Z.shape[0])
    for i in range(Z.shape[0]):
        max[i] = np.max(Z[i])
        Z_norm[i] = Z[i] - np.max(Z[i])
    return Z_norm, max

def batch_logsumeps(Z):
    Z_norm, max = batch_normlize(Z)
    lse = np.zeros(Z.shape[0])
    for i in range(Z_norm.shape[0]):
        ln = np.log(sum(np.exp(Z_norm[i])))
        lse[i] = ln + max[i]
    return lse

def batch_neg_log_probs(Z):
    lse = batch_logsumeps(Z)
    neg_log_probs = np.zeros_like(Z, dtype=float)
    for i in range(neg_log_probs.shape[0]):
        for j in range(neg_log_probs.shape[1]):
            neg_log_probs[i,j] = lse[i] - Z[i,j]
    return neg_log_probs

def batch_softmax(Z):
    Z_norm, max = batch_normlize(Z)
    res = np.zeros_like(Z_norm, dtype=float)
    for i in range(res.shape[0]):
        sum = np.sum(np.exp(Z_norm[i]))
        for j in range(res.shape[1]):
            res[i,j] = np.exp(Z_norm[i,j])/sum
    return res

def batch_loss(neg_log_probs, y):
    ave_loss = np.zeros(neg_log_probs.shape[0])
    for i in range(neg_log_probs.shape[0]):
        ave_loss[i] = neg_log_probs[i][y[i]]
    ave = np.sum(ave_loss) / neg_log_probs.shape[0]
    return ave_loss, ave

Z = np.array([[2,1,0],
              [0,0,0],
              [1000,999,998]])
y = np.array([0, 2, 1])

print(batch_normlize(Z),"\n")
print(batch_logsumeps(Z),"\n")
print(batch_neg_log_probs(Z),"\n")
print(batch_loss(batch_neg_log_probs(Z), y),"\n")
print(batch_softmax(Z))

(array([[ 0., -1., -2.],
       [ 0.,  0.,  0.],
       [ 0., -1., -2.]]), array([   2.,    0., 1000.])) 

[   2.40760596    1.09861229 1000.40760596] 

[[0.40760596 1.40760596 2.40760596]
 [1.09861229 1.09861229 1.09861229]
 [0.40760596 1.40760596 2.40760596]] 

(array([0.40760596, 1.09861229, 1.40760596]), np.float64(0.9712747391856397)) 

[[0.66524096 0.24472847 0.09003057]
 [0.33333333 0.33333333 0.33333333]
 [0.66524096 0.24472847 0.09003057]]


In [5]:
# Batch Softmax + 稳定交叉熵独立验收区
tol = 1e-12

P_actual = batch_softmax(Z)
neg_log_probs_actual = batch_neg_log_probs(Z)
losses_actual, mean_actual = batch_loss(neg_log_probs_actual, y)

# 1. 每行概率和
row_sum_residuals = np.abs(np.sum(P_actual, axis=1) - 1.0)
row_sum_max_residual = np.max(row_sum_residuals)
row_sum_pass = row_sum_max_residual < tol

# 2. Batch Softmax 与逐行一维 Softmax 对拍
P_reference = np.vstack([onedim_softmax(row) for row in Z])
softmax_max_residual = np.max(np.abs(P_actual - P_reference))
softmax_pass = softmax_max_residual < tol

# 3. 逐样本损失与理论值对拍
losses_reference = np.array([
    0.4076059644443804,
    1.0986122886681098,
    1.4076059644443804,
])
losses_max_residual = np.max(np.abs(losses_actual - losses_reference))
losses_pass = losses_max_residual < tol

# 4. Batch 平均损失
mean_reference = np.mean(losses_reference)
mean_residual = np.abs(mean_actual - mean_reference)
mean_pass = mean_residual < tol

# 5. 数值有限性与概率非负性
finite_pass = (
    np.all(np.isfinite(P_actual))
    and np.all(np.isfinite(losses_actual))
    and np.isfinite(mean_actual)
)
nonnegative_pass = np.all(P_actual >= 0.0)

all_pass = (
    row_sum_pass
    and softmax_pass
    and losses_pass
    and mean_pass
    and finite_pass
    and nonnegative_pass
)

print("逐行概率和：", np.sum(P_actual, axis=1))
print("逐行概率和残差：", row_sum_residuals)
print("概率和最大残差：", row_sum_max_residual, "PASS" if row_sum_pass else "FAIL")
print("Batch vs 一维 Softmax 最大残差：", softmax_max_residual, "PASS" if softmax_pass else "FAIL")
print("逐样本损失理论值：", losses_reference)
print("逐样本损失实测值：", losses_actual)
print("逐样本损失最大残差：", losses_max_residual, "PASS" if losses_pass else "FAIL")
print("平均损失理论值：", mean_reference)
print("平均损失实测值：", mean_actual)
print("平均损失绝对残差：", mean_residual, "PASS" if mean_pass else "FAIL")
print("数值全部有限：", finite_pass, "PASS" if finite_pass else "FAIL")
print("概率全部非负：", nonnegative_pass, "PASS" if nonnegative_pass else "FAIL")
print("总验收：", "PASS" if all_pass else "FAIL")


逐行概率和： [1. 1. 1.]
逐行概率和残差： [1.11022302e-16 0.00000000e+00 1.11022302e-16]
概率和最大残差： 1.1102230246251565e-16 PASS
Batch vs 一维 Softmax 最大残差： 0.0 PASS
逐样本损失理论值： [0.40760596 1.09861229 1.40760596]
逐样本损失实测值： [0.40760596 1.09861229 1.40760596]
逐样本损失最大残差： 4.8183679268731794e-14 PASS
平均损失理论值： 0.9712747391856235
平均损失实测值： 0.9712747391856397
平均损失绝对残差： 1.6209256159527285e-14 PASS
数值全部有限： True PASS
概率全部非负： True PASS
总验收： PASS


In [6]:
import numpy as np

# 解析梯度独立验收：刻意使用 B=2、C=3，避免把 batch 轴和类别轴写反却未暴露
P_test = np.array([[0.7, 0.2, 0.1],
                   [0.1, 0.3, 0.6]])
y_test = np.array([0, 2])

def label_2_onehot(P, label):
    Y = np.zeros_like(P, dtype=float)
    for i in range(len(label)):
        Y[i, label[i]] = 1.0
    return Y

def batch_ce_grad(P, y):
    return (P - label_2_onehot(P, y))/P.shape[0]

tol = 1e-12
P_before = P_test.copy()
dZ_expected = np.array([[-0.15, 0.10,  0.05],
                        [ 0.05, 0.15, -0.20]])
dZ_actual = batch_ce_grad(P_test, y_test)

# 1. 形状与理论矩阵对拍
shape_pass = dZ_actual.shape == dZ_expected.shape == (2, 3)
gradient_max_residual = np.max(np.abs(dZ_actual - dZ_expected))
gradient_pass = gradient_max_residual < tol

# 2. 每行梯度和应为 0
row_sums = np.sum(dZ_actual, axis=1)
row_sum_max_residual = np.max(np.abs(row_sums))
row_sum_pass = row_sum_max_residual < tol

# 3. 真实类梯度为负，非真实类梯度为正
target_gradients = dZ_actual[np.arange(P_test.shape[0]), y_test]
target_negative_pass = np.all(target_gradients < 0.0)
non_target_mask = np.ones_like(dZ_actual, dtype=bool)
non_target_mask[np.arange(P_test.shape[0]), y_test] = False
non_target_gradients = dZ_actual[non_target_mask]
non_target_positive_pass = np.all(non_target_gradients > 0.0)

# 4. 数值有限，且函数没有原地修改输入 P
finite_pass = np.all(np.isfinite(dZ_actual))
input_unchanged_pass = np.array_equal(P_test, P_before)

all_pass = (
    shape_pass
    and gradient_pass
    and row_sum_pass
    and target_negative_pass
    and non_target_positive_pass
    and finite_pass
    and input_unchanged_pass
)

print("one-hot 标签：\n", label_2_onehot(P_test, y_test))
print("理论梯度：\n", dZ_expected)
print("实测梯度：\n", dZ_actual)
print("输出形状：", dZ_actual.shape, "PASS" if shape_pass else "FAIL")
print("梯度最大绝对残差：", gradient_max_residual, "PASS" if gradient_pass else "FAIL")
print("每行梯度和：", row_sums)
print("行和最大绝对残差：", row_sum_max_residual, "PASS" if row_sum_pass else "FAIL")
print("真实类梯度：", target_gradients, "PASS" if target_negative_pass else "FAIL")
print("非真实类梯度：", non_target_gradients, "PASS" if non_target_positive_pass else "FAIL")
print("数值全部有限：", finite_pass, "PASS" if finite_pass else "FAIL")
print("输入 P 未被修改：", input_unchanged_pass, "PASS" if input_unchanged_pass else "FAIL")
print("总验收：", "PASS" if all_pass else "FAIL")


one-hot 标签：
 [[1. 0. 0.]
 [0. 0. 1.]]
理论梯度：
 [[-0.15  0.1   0.05]
 [ 0.05  0.15 -0.2 ]]
实测梯度：
 [[-0.15  0.1   0.05]
 [ 0.05  0.15 -0.2 ]]
输出形状： (2, 3) PASS
梯度最大绝对残差： 2.7755575615628914e-17 PASS
每行梯度和： [-1.38777878e-17  0.00000000e+00]
行和最大绝对残差： 1.3877787807814457e-17 PASS
真实类梯度： [-0.15 -0.2 ] PASS
非真实类梯度： [0.1  0.05 0.05 0.15] PASS
数值全部有限： True PASS
输入 P 未被修改： True PASS
总验收： PASS


In [7]:
# Softmax + 平均交叉熵：解析梯度 vs 中心差分数值梯度
def mean_loss_from_logits(Z, y):
    neg_log_probs = batch_neg_log_probs(Z)
    _, mean_loss = batch_loss(neg_log_probs, y)
    return float(mean_loss)

def numerical_gradient(Z, y, eps=1e-5):
    dZ_numerical = np.zeros_like(Z, dtype=float)
    for i in range(Z.shape[0]):
        for j in range(Z.shape[1]):
            Z_plus = Z.copy()
            Z_minus = Z.copy()
            Z_plus[i, j] += eps
            Z_minus[i, j] -= eps
            loss_plus = mean_loss_from_logits(Z_plus, y)
            loss_minus = mean_loss_from_logits(Z_minus, y)
            dZ_numerical[i, j] = (loss_plus - loss_minus)/(2.0*eps)
    return dZ_numerical

Z_grad_test = np.array([[2.0, 1.0, 0.0],
                        [0.0, 0.0, 0.0],
                        [1000.0, 999.0, 998.0]])
y_grad_test = np.array([0, 2, 1])
eps = 1e-5
tol = 1e-8
Z_before = Z_grad_test.copy()

P_grad_test = batch_softmax(Z_grad_test)
dZ_analytic = batch_ce_grad(P_grad_test, y_grad_test)
dZ_numerical = numerical_gradient(Z_grad_test, y_grad_test, eps=eps)
residual_matrix = np.abs(dZ_analytic - dZ_numerical)
max_residual = np.max(residual_matrix)

shape_pass = dZ_analytic.shape == dZ_numerical.shape == Z_grad_test.shape
residual_pass = max_residual < tol
finite_pass = np.all(np.isfinite(dZ_analytic)) and np.all(np.isfinite(dZ_numerical))
input_unchanged_pass = np.array_equal(Z_grad_test, Z_before)
all_pass = shape_pass and residual_pass and finite_pass and input_unchanged_pass

print("解析梯度：\n", dZ_analytic)
print("数值梯度：\n", dZ_numerical)
print("绝对残差矩阵：\n", residual_matrix)
print("最大绝对残差：", max_residual)
print("允差：", tol, "PASS" if residual_pass else "FAIL")
print("形状一致：", shape_pass, "PASS" if shape_pass else "FAIL")
print("数值全部有限：", finite_pass, "PASS" if finite_pass else "FAIL")
print("输入 Z 未被修改：", input_unchanged_pass, "PASS" if input_unchanged_pass else "FAIL")
print("总验收：", "PASS" if all_pass else "FAIL")


解析梯度：
 [[-0.11158635  0.08157616  0.03001019]
 [ 0.11111111  0.11111111 -0.22222222]
 [ 0.22174699 -0.25175718  0.03001019]]
数值梯度：
 [[-0.11158635  0.08157616  0.03001019]
 [ 0.11111111  0.11111111 -0.22222222]
 [ 0.22174699 -0.25175718  0.03001019]]
绝对残差矩阵：
 [[2.32526498e-12 5.50211265e-12 3.17688931e-12]
 [7.27903848e-13 7.27903848e-13 1.45577994e-12]
 [7.87216181e-10 9.69310410e-10 1.36403652e-10]]
最大绝对残差： 9.693104097152627e-10
允差： 1e-08 PASS
形状一致： True PASS
数值全部有限： True PASS
输入 Z 未被修改： True PASS
总验收： PASS
